In [33]:
import duckdb
import pandas as pd

In [34]:
con = duckdb.connect()

In [35]:
con.sql("""
CREATE SECRET (
    TYPE huggingface
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [36]:
import duckdb

print("DuckDB Version:", duckdb.__version__)

DuckDB Version: 1.5.4


In [37]:
con.sql("""
INSTALL httpfs;
LOAD httpfs;
""")

In [38]:
con.sql("""
SELECT *
FROM duckdb_secrets();
""")

┌───────────────────────┬─────────────┬──────────┬────────────┬─────────┬───────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         name          │    type     │ provider │ persistent │ storage │   scope   │                                              secret_string                                               │
│        varchar        │   varchar   │ varchar  │  boolean   │ varchar │ varchar[] │                                                 varchar                                                  │
├───────────────────────┼─────────────┼──────────┼────────────┼─────────┼───────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ __default_huggingface │ huggingface │ config   │ false      │ memory  │ ['hf://'] │ name=__default_huggingface;type=huggingface;provider=config;serializable=true;scope=hf://;token=redacted │
└───────────────────────┴──────────

In [39]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [40]:
first = next(iter(ds))
print(first)

{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}


In [41]:
first = next(iter(ds))

print("Number of columns:", len(first))
print(first.keys())

Number of columns: 30
dict_keys(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'])


In [42]:
import pandas as pd

sample = list(ds.take(1000))
df = pd.DataFrame(sample)

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [43]:
print("Earliest date :", df["report_date"].min())
print("Latest date   :", df["report_date"].max())

Earliest date : 2025-01-27
Latest date   : 2025-01-30


## 1. Unit of Analysis

**Unit of Analysis**

Each row represents the daily performance of one content item for one client on one reporting date.

**Time Window**

The dataset contains daily observations over the reporting period available in the FlyRank Internship Warehouse release. The exact date range is verified by the code cells below.

The data originates from the FlyRank warehouse, where Google Search Console and Google Analytics data are aggregated, pseudonymized, and released as a safe research dataset.

In [44]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [45]:
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing

report_date                 0
client_hash_id              0
ai_other                    0
ai_meta                     0
ai_claude                   0
ai_copilot                  0
ai_gemini                   0
ai_perplexity               0
ai_chatgpt                  0
sessions_ai                 0
sessions_paid               0
sessions_social             0
sessions_referral           0
sessions_direct             0
sessions_organic            0
ga4_total_engagement_sec    0
ga4_engaged_sessions        0
ga4_users                   0
ga4_sessions                0
ga4_pageviews               0
gsc_avg_position            0
gsc_sum_position            0
gsc_clicks                  0
gsc_impressions             0
ga4_data_available          0
gsc_data_available          0
client_has_ga4              0
client_has_gsc              0
content_hash_id             0
scroll_events               0
dtype: int64

In [46]:
df[
    ["report_date", "client_hash_id", "content_hash_id"]
].duplicated().sum()

0

In [47]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

 1. report_date
 2. client_hash_id
 3. content_hash_id
 4. client_has_gsc
 5. client_has_ga4
 6. gsc_data_available
 7. ga4_data_available
 8. gsc_impressions
 9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events


# 2. Fields

## Feature Fields

The following fields can be used as model features because they describe search performance, website traffic, and user engagement.

### Google Search Console (GSC)
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position

### Google Analytics 4 (GA4)
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec

### Traffic Sources
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai

### AI Traffic
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_claude
- ai_meta
- ai_other

### Engagement
- scroll_events


---

## Label / Proxy

This dataset does not contain an explicit ML label.

For this exploratory data contract, Google Search Console clicks (gsc_clicks) are treated as a proxy outcome variable because they measure user interaction with search results.

This proxy could later be replaced depending on the downstream ML task.


---

## Context Fields

These fields identify records or describe data availability.

- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available


---

## Excluded Fields

The following fields are excluded as predictive features.

- client_hash_id
- content_hash_id

These are pseudonymized identifiers used only for joining records and verifying dataset grain. They should not be used as model inputs because they do not contain meaningful predictive information.

In [48]:
print("Rows :", len(df))
print("Columns :", len(df.columns))

Rows : 1000
Columns : 30


In [49]:
duplicates = df[
    ["report_date", "client_hash_id", "content_hash_id"]
].duplicated().sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [50]:
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing

report_date                 0
client_hash_id              0
ai_other                    0
ai_meta                     0
ai_claude                   0
ai_copilot                  0
ai_gemini                   0
ai_perplexity               0
ai_chatgpt                  0
sessions_ai                 0
sessions_paid               0
sessions_social             0
sessions_referral           0
sessions_direct             0
sessions_organic            0
ga4_total_engagement_sec    0
ga4_engaged_sessions        0
ga4_users                   0
ga4_sessions                0
ga4_pageviews               0
gsc_avg_position            0
gsc_sum_position            0
gsc_clicks                  0
gsc_impressions             0
ga4_data_available          0
gsc_data_available          0
client_has_ga4              0
client_has_gsc              0
content_hash_id             0
scroll_events               0
dtype: int64

In [51]:
print("Earliest:", df.report_date.min())
print("Latest:", df.report_date.max())

Earliest: 2025-01-27
Latest: 2025-01-30


In [52]:
df[
    [
        "client_has_gsc",
        "client_has_ga4",
        "gsc_data_available",
        "ga4_data_available"
    ]
].describe()

,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available
count,1000,1000,1000,1000
unique,1,1,1,1
top,True,True,True,False
freq,1000,1000,1000,1000


# 4. Data Limits

This notebook uses a streamed sample of the FlyRank Internship Warehouse dataset rather than the complete dataset.

The sample is sufficient for validating the data contract but does not necessarily represent the full distribution of records.

The dataset is pseudonymized. Client identifiers and content identifiers cannot be mapped back to real organizations or webpages.

No personally identifiable information (PII) is included.

The dataset should only be used for research and educational purposes according to the FlyRank dataset terms.

# 5. Self Check

✅ Unit of analysis identified

✅ Time window verified

✅ Feature fields documented

✅ Context fields documented

✅ Excluded fields documented

✅ Label/proxy identified

✅ Missing values checked

✅ Grain verified

✅ Date window verified

✅ Safe fields only

✅ Data limitations documented